**IIC3912 - Tópicos Avanzados de Gráfica Computacional**\
Francisca T. Gil-Ureta ⋅ 2026

# **Clase 3 | Offline Mesh Processing**

## Mallas Poligonales cómo datos

In [1]:
!pip install trimesh

import sys
from pathlib import Path
import numpy as np

# plotly es una librería de plotting con buen soporte para interactive & 3D
import plotly.graph_objects as go

# trimesh es una libreria para cargar y manipular triangle meshes
import trimesh

In [2]:
path = Path("bunny.obj")
if 'google.colab' in sys.modules:
    path = Path("/content/bunny.obj")
if not path.exists():
    ! wget "https://graphics.stanford.edu/~mdfisher/Data/Meshes/bunny.obj"

## Obj File Format
El formato OBJ es uno de los formatos más comunes para almacenar mallas 3D.

**Características principales**

* Formato basado en texto (ASCII)
* Fácil de leer y editar manualmente
* Representa principalmente geometría de mallas poligonales
* Puede incluir referencias a materiales (archivo .MTL)

In [3]:
! cat /content/bunny.obj | head -n 10

cat: /content/bunny.obj: No such file or directory


In [4]:
mesh = trimesh.load(path, process=False)
mesh.show()

ValueError: string is not a file: `/home/gilureta/Development/grafica-2026/praticals/practical_03/data/bunny.obj`

OBJ es un formato simple, para el cual podemos escribir nuestro propio read/write function.

In [ ]:
def read_obj(filename: Path):
    """
    Reads the simplest obj file which only contain vertices and faces.
    Does not read normals nor uv information.
    """
    vertices = []
    faces = []

    with filename.open("r") as fin:
        for line in fin.readlines():
            if line.startswith('#'):
                continue
            values = line.split()
            if not values:
                continue

            if values[0] == "v":
                vertices.append([float(x) for x in values[1:4]])
            elif values[0] == "f":
                faces.append([int(x) for x in values[1:4]])

    return np.array(vertices), np.array(faces) - 1

In [ ]:
# leemos el archivo
vertices, faces = read_obj(path)

# Podemos dibujar las caras usando Mesh3d
# https://plotly.com/python/3d-mesh/
x, y, z = vertices.T
i, j, k = faces.T

data = go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k)
figure = go.Figure(data=data,layout=go.Layout(scene=dict(aspectmode='data')))
figure.update_layout(margin=dict(l=0, r=0, t=0, b=0), scene_camera=dict(up=dict(x=0.0, y=1.0, z=0.0)))
figure.show()

## Ejercicio
* Basado en en lo que vez, es esta malla watertight?
* Cuántos vertices y faces tiene el mesh?
* Que tipos de arreglos son usados para los vertices y faces?


In [ ]:
# answer
print(faces.shape)
print(faces.dtype)

## Ejercicio

*   Cuales son los 3 vertices de la decima cara
*   Cuales son la posición 3D de esos vertices



In [ ]:
v0, v1, v2 = faces[9,:]
print(v0, v1, v2)
vertices[v0, :]

vertices[faces[9,:],:]

# Ejercicio
Calcula si el mesh tiene _'dangling vertices'_ (i.e., si hay vertices que no son usados por ningun mesh)

In [ ]:
# answer
unique_vertices = np.unique(faces)
print(unique_vertices.shape)
print(vertices.shape)
print(np.min(unique_vertices), np.max(unique_vertices))

## Ejercicio - Edges
- Cómo podemos calcular si el mesh es watertight?
- Cómo obtenemos los boundary edges (aristas que están en el borde)

In [ ]:
# GET ALL EDGES
edges = []
for i, (v0, v1, v2) in enumerate(mesh.faces):
    edges.append((v0, v1))
    edges.append((v1, v2))
    edges.append((v2, v0))
edges = np.array(edges)
print(len(edges))

# SORT THE EDGES
edges_sorted = np.array([ (min(x,y), max(x,y)) for x, y in edges])

# UNIQUE EDGES
edges_unique, counts = np.unique(edges_sorted, axis=0, return_counts=True)
print(len(edges_unique))

In [ ]:
edges_borde = edges_unique[counts == 1]

## Ejercicio AABB
Cómo calculamos el Alix Aligned Bounding Box del modelo?
- bounding box Extents: el largo de la caja en las 3 dimensiones
- bounding box Center: es el centro de la caja en 3D

In [ ]:
maxv = np.max(mesh.vertices, axis=0)
minv = np.min(mesh.vertices, axis=0)
extents = maxv - minv
center = (maxv + minv) / 2.0
print("extent", extents)
print("center", center)

## Ejercicio
Visualizar los elementos problemáticos y el bounding box

In [ ]:
# visualizar lo que teníamos antes pero agregamos los edges

# para agregar lineas en plotly3D usamos Scatter3d. Cada segmento AB debe ser
# agregado como [[A_x, A_y, A_z], [B_x, B_y, B_z], [None, None, None]]
# los None indican que la linea es discontinua

edges_lines = []
for e in edges_borde:
  edges_lines.append(mesh.vertices[e[0]])
  edges_lines.append(mesh.vertices[e[1]])
  edges_lines.append([None, None, None])
edges_lines = np.array(edges_lines)
data_edges = go.Scatter3d(x=edges_lines[:,0], y=edges_lines[:,1], z=edges_lines[:,2], mode="lines")

# podemos agregar el bounding box usando Scatter3D
# TODO:

# agregamos también el mesh original
x, y, z = vertices.T
i, j, k = faces.T
data_mesh = go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k)

data = [data_mesh, data_edges]

figure = go.Figure(data=data,layout=go.Layout(scene=dict(aspectmode='data')))
figure.update_layout(margin=dict(l=0, r=0, t=0, b=0), scene_camera=dict(up=dict(x=0.0, y=1.0, z=0.0)))
figure.show()